## 8. Summary & Resources

### What We Covered
1. **Crystal → Graph**: How crystal structures are converted to graphs (atoms=nodes, bonds=edges, cutoff radius, periodic boundary conditions)
2. **MEGNet**: Trained a materials graph network to predict formation energies
3. **Pretrained Models**: Used M3GNet for instant property prediction and structural relaxation
4. **Benchmarks**: The MatBench ecosystem for evaluating crystal property prediction models

### Key Takeaways
- GNNs are now the **state of the art** for crystal property prediction, outperforming hand-crafted descriptors
- **Pretrained universal potentials** (M3GNet, CHGNet, MACE) are game-changers — predict properties for ANY crystal without retraining
- The field is moving fast: equivariant models (MACE, NequIP) capture more physics by respecting rotational symmetry

### GitHub Repositories

| Repository | Stars | Description |
|-----------|-------|-------------|
| [`txie-93/cgcnn`](https://github.com/txie-93/cgcnn) | ~1000 | Original CGCNN implementation |
| [`materialsvirtuallab/matgl`](https://github.com/materialsvirtuallab/matgl) | ~300 | MEGNet & M3GNet in PyTorch (recommended) |
| [`usnistgov/alignn`](https://github.com/usnistgov/alignn) | ~200 | ALIGNN with bond angle features |
| [`ACEsuit/mace`](https://github.com/ACEsuit/mace) | ~1000+ | MACE equivariant potential |
| [`mir-group/nequip`](https://github.com/mir-group/nequip) | ~825 | E(3)-equivariant interatomic potentials |
| [`atomistic-machine-learning/schnetpack`](https://github.com/atomistic-machine-learning/schnetpack) | ~800+ | SchNet, PaiNN, and more |

### Datasets

| Dataset | Size | Access |
|---------|------|--------|
| Materials Project | ~150k+ materials | https://materialsproject.org (free API) |
| MatBench | 13 benchmark tasks | `pip install matbench` |
| JARVIS-DFT | ~76k materials | https://jarvis.nist.gov |
| Open Catalyst (OC20) | ~250M DFT calcs | https://opencatalystproject.org |

### Key Papers
1. Xie & Grossman. "Crystal Graph Convolutional Neural Networks..." PRL 120, 145301 (2018)
2. Chen et al. "Graph Networks as a Universal ML Framework..." Chem. Mater. 31, 3564 (2019)
3. Choudhary & DeCost. "Atomistic Line Graph Neural Network..." npj Comp. Mat. 7, 185 (2021)
4. Chen & Ong. "A Universal Graph Deep Learning Interatomic Potential..." Nature Comp. Sci. 2, 718 (2022)
5. Batatia et al. "MACE: Higher Order Equivariant Message Passing..." NeurIPS 2022
6. Dunn et al. "Benchmarking Materials Property Prediction Methods..." npj Comp. Mat. 6, 138 (2020)
7. Riebesell et al. "Matbench Discovery..." Nature Mach. Intell. 7, 836 (2025)

## 7. The Bigger Picture: Benchmarks & State of the Art

### MatBench Leaderboard

The **MatBench** benchmark (Dunn et al., npj Comp. Mat. 2020) standardizes evaluation of ML models for materials property prediction. Key tasks include:

| Task | Property | # Samples | Best MAE |
|------|----------|-----------|----------|
| `matbench_mp_e_form` | Formation energy | 132,752 | ~0.020 eV/atom |
| `matbench_mp_gap` | Band gap | 106,113 | ~0.14 eV |
| `matbench_log_kvrh` | Bulk modulus (log) | 10,987 | ~0.046 |
| `matbench_log_gvrh` | Shear modulus (log) | 10,987 | ~0.065 |
| `matbench_perovskites` | Formation energy | 18,928 | ~0.027 eV |

### MatBench Discovery (Crystal Stability)

For the harder task of predicting crystal *stability* (will this structure survive?), the MatBench Discovery leaderboard ranks 45+ models:

**Top performers (F1 score):** PET-OAM-XL (0.924) > CHGNet > M3GNet > MACE > ALIGNN > MEGNet > CGCNN

This hierarchy reflects increasing model sophistication — but even CGCNN remains competitive for many tasks!

Explore the leaderboards:
- MatBench: https://matbench.materialsproject.org
- MatBench Discovery: https://matbench-discovery.materialsproject.org

In [ ]:
# Load the M3GNet Universal Potential for structural relaxation
from matgl.ext.ase import M3GNetCalculator
from pymatgen.io.ase import AseAtomsAdaptor

pot = matgl.load_model("M3GNet-MP-2021.2.8-PES")
calc = M3GNetCalculator(potential=pot)

# Relax a slightly distorted NaCl structure
distorted_nacl = Structure(
    Lattice.cubic(5.80),  # slightly expanded
    ["Na", "Cl"],
    [[0.02, 0.01, 0.0], [0.48, 0.51, 0.50]]  # slightly off ideal positions
)

atoms = AseAtomsAdaptor.get_atoms(distorted_nacl)
atoms.calc = calc

print("Before relaxation:")
print(f"  Energy: {atoms.get_potential_energy():.4f} eV")
print(f"  Lattice parameter: {distorted_nacl.lattice.a:.3f} Å")
print(f"  Max force: {np.max(np.abs(atoms.get_forces())):.4f} eV/Å")

# Perform relaxation
from ase.optimize import BFGS
import io

optimizer = BFGS(atoms, logfile=io.StringIO())
optimizer.run(fmax=0.05, steps=50)

relaxed_structure = AseAtomsAdaptor.get_structure(atoms)
print(f"\nAfter relaxation:")
print(f"  Energy: {atoms.get_potential_energy():.4f} eV")
print(f"  Lattice parameter: {relaxed_structure.lattice.a:.3f} Å")
print(f"  Max force: {np.max(np.abs(atoms.get_forces())):.4f} eV/Å")

In [ ]:
# Load pretrained MEGNet model for formation energy
pretrained_model = matgl.load_model("MEGNet-MP-2019.4.1-Eform")
print("Loaded pretrained MEGNet-MP-2019.4.1-Eform")
print(f"Trained on: Materials Project 2019.4.1 release")
print(f"Target: Formation energy per atom (eV/atom)")

# Predict formation energies for some well-known materials
test_structures = {
    "NaCl": Structure(Lattice.cubic(5.64), ["Na", "Cl"], [[0,0,0], [0.5,0.5,0.5]]),
    "MgO": Structure(Lattice.cubic(4.21), ["Mg", "O"], [[0,0,0], [0.5,0.5,0.5]]),
    "Si": Structure(Lattice.cubic(5.43), ["Si", "Si"], [[0,0,0], [0.25,0.25,0.25]]),
    "GaAs": Structure(Lattice.cubic(5.65), ["Ga", "As"], [[0,0,0], [0.25,0.25,0.25]]),
    "BaTiO3": Structure(Lattice.cubic(4.01), ["Ba", "Ti", "O", "O", "O"],
                        [[0,0,0], [0.5,0.5,0.5], [0.5,0.5,0], [0.5,0,0.5], [0,0.5,0.5]]),
}

# Known DFT values (approximate, from Materials Project)
known_eform = {"NaCl": -1.89, "MgO": -3.08, "Si": -0.006, "GaAs": -0.34, "BaTiO3": -3.36}

print(f"\n{'Material':<10} {'Predicted':>12} {'DFT (approx)':>14} {'Error':>10}")
print("-" * 50)

for name, structure in test_structures.items():
    eform_pred = pretrained_model.predict_structure(structure)
    eform_pred_val = float(eform_pred)
    eform_dft = known_eform[name]
    error = abs(eform_pred_val - eform_dft)
    print(f"{name:<10} {eform_pred_val:>12.4f} {eform_dft:>14.4f} {error:>10.4f}")

## 6. Using Pretrained Models

One of the biggest advantages of `matgl` is access to **pretrained universal models**. These were trained on the entire Materials Project dataset and can predict properties for any crystal structure instantly — no training required!

The **M3GNet Universal Potential** can:
- Predict energies, forces, and stresses
- Perform structural relaxation
- Run molecular dynamics simulations

In [ ]:
# Evaluate on test set — parity plot (predicted vs actual)
import torch

model.eval()
predictions = []
actuals = []

for batch in test_loader:
    graph, labels_batch, attrs = batch
    preds = model(graph, attrs)
    predictions.extend(preds.detach().cpu().numpy().flatten().tolist())
    actuals.extend(labels_batch.detach().cpu().numpy().flatten().tolist())

predictions = np.array(predictions)
actuals = np.array(actuals)

mae = np.mean(np.abs(predictions - actuals))
rmse = np.sqrt(np.mean((predictions - actuals)**2))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(actuals, predictions, alpha=0.5, s=20, color='steelblue')
lims = [min(actuals.min(), predictions.min()) - 0.2, 
        max(actuals.max(), predictions.max()) + 0.2]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel("DFT Formation Energy (eV/atom)", fontsize=13)
ax.set_ylabel("MEGNet Predicted (eV/atom)", fontsize=13)
ax.set_title(f"MEGNet Predictions\nMAE = {mae:.3f} eV/atom, RMSE = {rmse:.3f} eV/atom", fontsize=14)
ax.legend(fontsize=12)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f"Mean Absolute Error: {mae:.4f} eV/atom")
print(f"RMSE: {rmse:.4f} eV/atom")
print(f"\nNote: State-of-the-art MEGNet achieves ~0.028 eV/atom MAE on the full MP dataset.")

## 5. Evaluation

Let's evaluate the trained model and visualize its predictions.

In [ ]:
# Train the model
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
print("Training complete!")

In [ ]:
# Define the MEGNet model
model = MEGNet(
    dim_node_embedding=16,
    dim_edge_embedding=100,
    dim_state_embedding=2,
    nblocks=3,
    hidden_layer_sizes_input=(64, 32),
    hidden_layer_sizes_conv=(64, 64, 32),
    nlayers_set2set=1,
    niters_set2set=2,
    hidden_layer_sizes_output=(32, 16),
    is_classification=False,
    activation_type="softplus2",
    element_types=elem_list,
    bond_expansion=BondExpansion(rbf_type="Gaussian", initial=0.0, final=5.0, num_centers=100, width=0.5),
)

print(f"MEGNet model created with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Element types: {elem_list}")

# Set up PyTorch Lightning trainer
lit_model = ModelLightningModule(model=model, lr=1e-3)

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    inference_mode=False,
)

print("\nStarting training...")

In [ ]:
# Set up the matgl MEGNet model and training pipeline
elem_list = get_element_list(structures)
converter = Structure2Graph(element_types=elem_list, cutoff=5.0)

# Create the matgl dataset
dataset = MGLDataset(
    threebody_cutoff=None,
    structures=structures,
    converter=converter,
    labels=labels,
    label_name="Eform",
)

# Split into train/val/test
train_data, val_data, test_data = split_dataset(
    dataset, 
    frac_list=[0.7, 0.15, 0.15],
    shuffle=True,
    random_state=42,
)

# Create data loaders
train_loader, val_loader, test_loader = MGLDataLoader(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    collate_fn=collate_fn_graph,
    batch_size=32,
    num_workers=0,
)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
# Build and train MEGNet using matgl's official approach
# Reference: matgl tutorial "Training a MEGNet Formation Energy Model"

from matgl.ext.pymatgen import Structure2Graph, get_element_list
from matgl.graph.data import MGLDataset, MGLDataLoader, collate_fn_graph
from matgl.models import MEGNet
from matgl.layers import BondExpansion
from matgl.utils.training import ModelLightningModule
import pytorch_lightning as pl
import dgl

# For the tutorial, we create a small synthetic dataset to ensure it runs
# In practice, you'd use the full MP dataset

# Create a set of example crystal structures with known formation energies
structures = []
eform_per_atom = []

# Simple cubic metals and binary compounds
example_data = [
    # (formula, space_group, lattice_param, positions, energy_per_atom)
    (["Si"], Lattice.cubic(5.43), [[0, 0, 0]], -0.006),
    (["Ge"], Lattice.cubic(5.66), [[0, 0, 0]], 0.0),
    (["Na", "Cl"], Lattice.cubic(5.64), [[0,0,0], [0.5,0.5,0.5]], -1.89),
    (["Li", "F"], Lattice.cubic(4.03), [[0,0,0], [0.5,0.5,0.5]], -3.18),
    (["Mg", "O"], Lattice.cubic(4.21), [[0,0,0], [0.5,0.5,0.5]], -3.08),
    (["Ca", "O"], Lattice.cubic(4.81), [[0,0,0], [0.5,0.5,0.5]], -3.17),
    (["Sr", "O"], Lattice.cubic(5.16), [[0,0,0], [0.5,0.5,0.5]], -3.07),
    (["Ba", "O"], Lattice.cubic(5.52), [[0,0,0], [0.5,0.5,0.5]], -2.79),
    (["Al"], Lattice.cubic(4.05), [[0,0,0]], 0.0),
    (["Cu"], Lattice.cubic(3.61), [[0,0,0]], 0.0),
    (["Fe"], Lattice.cubic(2.87), [[0,0,0]], 0.0),
    (["Ti"], Lattice.hexagonal(2.95, 4.68), [[1/3,2/3,1/4]], 0.0),
    (["Zn", "O"], Lattice.hexagonal(3.25, 5.21), [[1/3,2/3,0], [1/3,2/3,0.38]], -1.72),
    (["Ga", "As"], Lattice.cubic(5.65), [[0,0,0], [0.25,0.25,0.25]], -0.34),
    (["Ga", "N"], Lattice.hexagonal(3.19, 5.19), [[1/3,2/3,0], [1/3,2/3,0.38]], -0.56),
]

for species, lattice, coords, eform in example_data:
    s = Structure(lattice, species, coords)
    # Create supercell variants to increase dataset size
    for _ in range(10):
        noise = np.random.normal(0, 0.02, size=(len(coords), 3))
        noisy_coords = np.array(coords) + noise
        s_noisy = Structure(lattice, species, noisy_coords)
        structures.append(s_noisy)
        eform_per_atom.append(eform + np.random.normal(0, 0.05))

labels = {"Eform": eform_per_atom}
print(f"Created {len(structures)} training structures")
print(f"Formation energy range: {min(eform_per_atom):.2f} to {max(eform_per_atom):.2f} eV/atom")

## 4. Training a MEGNet Model

**MEGNet** (Materials Graph Network) uses the following architecture:
1. **Node embedding**: Encode element identity into a learned vector
2. **Edge embedding**: Gaussian expansion of interatomic distances
3. **Message passing blocks**: Update node, edge, and global state features iteratively
4. **Set2Set readout**: Aggregate node features into a single graph-level prediction
5. **Output MLP**: Map the graph representation to the target property

The `matgl` library makes this straightforward with PyTorch Lightning integration.

In [ ]:
# Load a dataset of structures + formation energies
# matgl includes a utility to load MP data for training
# We use a JSON dataset that ships with matgl examples

from matgl.utils.training import ModelLightningModule
import json, os

# Download a small dataset from Materials Project via matgl
# This uses the mp_api client. If you don't have an API key, 
# matgl provides cached datasets in its examples.

# For this tutorial, let's create a dataset from the matgl example data
# The matgl repo includes a dataset of ~5,000 structures for demos

try:
    # Try loading from matgl's built-in example data
    from matgl.ext.pymatgen import Structure2Graph
    from dgl.data.utils import split_dataset
    import dgl
    
    # Load the MP formation energy dataset
    # matgl uses a JSON format with structures and energies
    dataset_url = "https://figshare.com/ndownloader/files/40344436"
    print("Loading Materials Project formation energy dataset...")
    print("(This may take a minute on first run)")
    
    from matgl.utils.io import RemoteFile
    
    # Alternative: use matgl's MEGNet training example approach
    # Load structures from the MPF dataset
    import urllib.request
    
    data_path = "mp.2018.6.1.json"
    if not os.path.exists(data_path):
        print("Downloading MP dataset (this is the standard matgl training set)...")
        urllib.request.urlretrieve(
            "https://figshare.com/ndownloader/files/40344436",
            data_path
        )
    
    with open(data_path) as f:
        mp_data = json.load(f)
    
    print(f"Total structures in dataset: {len(mp_data)}")
    print(f"Sample entry keys: {list(mp_data[0].keys()) if isinstance(mp_data, list) else list(mp_data.keys())[:5]}")
    
except Exception as e:
    print(f"Note: Could not load full dataset ({e})")
    print("We'll use a smaller manually-created dataset for the demo.")

In [ ]:
import matgl
from matgl.ext.pymatgen import get_element_list

# matgl provides a curated MP 2018 dataset for formation energy training
# For the tutorial, we'll use the MPF.2021.2.8 dataset (a subset)
# This ships with matgl and is used in their official tutorials

from matgl.utils.io import get_available_pretrained_models
print("Available pretrained models in matgl:")
for m in sorted(get_available_pretrained_models()):
    print(f"  {m}")

## 3. Loading Materials Data

We'll use the **Materials Project** dataset — one of the largest open databases of computed material properties. The `matgl` library provides convenient access to curated subsets.

**Formation energy** (`E_form`) is the energy gained/lost when a material forms from its constituent elements. It's the most standard prediction target:
- Negative = thermodynamically stable
- Positive = metastable or unstable
- Units: eV/atom

In [ ]:
# Visualize the graph construction process
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: show the crystal structure (2D projection)
ax = axes[0]
coords = nacl.cart_coords
species = [str(s) for s in nacl.species]
colors = ['blue' if s == 'Na' else 'green' for s in species]
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=200, zorder=5, edgecolors='black')
for i, s in enumerate(species):
    ax.annotate(s, (coords[i, 0], coords[i, 1]), ha='center', va='center', 
                fontweight='bold', color='white', fontsize=10)
ax.set_title("Unit Cell (2D projection)", fontsize=14)
ax.set_xlabel("x (Å)")
ax.set_ylabel("y (Å)")
ax.set_aspect('equal')

# Right: show the graph (adjacency)
ax = axes[1]
# Draw nodes
node_pos = {0: (0, 0), 1: (1, 1)}
for idx, (pos, spec) in enumerate(zip(node_pos.values(), species)):
    color = 'royalblue' if spec == 'Na' else 'forestgreen'
    ax.scatter(*pos, s=600, c=color, zorder=5, edgecolors='black', linewidths=2)
    ax.annotate(spec, pos, ha='center', va='center', fontweight='bold', 
                color='white', fontsize=12)

# Draw edges with distance labels
from pymatgen.core.structure import PeriodicSite
nn_dists = sorted(set(round(n.nn_distance, 2) for n in neighbors[0]))[:3]
for i, d in enumerate(nn_dists):
    ax.annotate(f"d={d} Å", (0.5, 0.5 - i*0.15), ha='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

ax.plot([0, 1], [0, 1], 'k-', linewidth=2, zorder=1)
ax.set_title("Crystal Graph Representation", fontsize=14)
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal')
ax.axis('off')

plt.tight_layout()
plt.show()
print("Each edge encodes the interatomic distance via a Gaussian basis expansion.")

In [ ]:
# Find neighbors within a cutoff radius — this is how crystal graphs are built
cutoff = 5.0  # Angstroms
neighbors = nacl.get_all_neighbors(r=cutoff)

print(f"Cutoff radius: {cutoff} Å")
for i, (site, nbrs) in enumerate(zip(nacl, neighbors)):
    print(f"\n{site.specie} at {site.frac_coords}:")
    print(f"  Number of neighbors: {len(nbrs)}")
    for nbr in sorted(nbrs, key=lambda x: x.nn_distance)[:5]:
        print(f"  → {nbr.species_string} at distance {nbr.nn_distance:.3f} Å")

In [ ]:
# Create a simple crystal structure: NaCl (rock salt)
lattice = Lattice.cubic(5.64)  # 5.64 Angstrom cubic lattice
nacl = Structure(lattice, ["Na", "Cl"], 
                 [[0, 0, 0], [0.5, 0.5, 0.5]])

print("NaCl Crystal Structure:")
print(nacl)
print(f"\nLattice parameters: a={nacl.lattice.a:.2f} Å")
print(f"Number of atoms in unit cell: {len(nacl)}")
print(f"Volume: {nacl.volume:.2f} ų")

## 2. Understanding Crystal Graphs

### How does a crystal become a graph?

1. **Start with a unit cell**: A crystal is defined by a lattice + atom positions
2. **Atoms → Nodes**: Each atom becomes a node. Features encode element identity (atomic number, electronegativity, radius, etc.)
3. **Bonds → Edges**: Two atoms are connected if they're within a **cutoff radius** `r_cut` (typically 4–6 Å). Edge features encode the **interatomic distance** (often via Gaussian basis expansion)
4. **Periodicity**: Atoms in neighboring periodic images are included as neighbors, so the graph captures the infinite crystal

```
Unit Cell → Periodic Structure → Graph
┌─────┐    ┌─────┬─────┬─────┐    ○──○
│ ○ ○ │ →  │ ○ ○ │ ○ ○ │ ○ ○ │ →  │╲ │╲
│ ○ ○ │    │ ○ ○ │ ○ ○ │ ○ ○ │    ○──○
└─────┘    └─────┴─────┴─────┘    (fully connected within r_cut)
```

Let's see this in action with a real crystal.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.core import Structure, Lattice, Element

print("Imports successful!")

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install matgl pymatgen torch pytorch-lightning matplotlib numpy pandas mp-api

## 1. Setup & Installation

We use the following key libraries:
- **`matgl`** — Materials Graph Library: PyTorch implementation of MEGNet and M3GNet with pretrained models
- **`pymatgen`** — Python Materials Genomics: crystal structure manipulation and Materials Project API
- **`pytorch-lightning`** — Simplified training loops

# Crystal Graph Neural Networks for Property Prediction

## AI4Physics Learning Workshop — Uppsala University, April 2026

**Tutorial:** Geometric Deep Learning: Hands-on

This notebook introduces **Graph Neural Networks (GNNs) for predicting properties of crystalline materials**. We will:

1. Understand how crystal structures are represented as graphs
2. Explore real materials data from the Materials Project
3. Train a MEGNet model to predict formation energies
4. Use pretrained universal models (M3GNet) for instant predictions

### Why Crystal Structure GNNs?

Crystals are the backbone of materials science — from semiconductors to superconductors, batteries to catalysts. Predicting material properties typically requires expensive quantum mechanical calculations (DFT). GNNs offer a fast, accurate alternative by learning directly from crystal structure.

**Key insight:** A crystal can be naturally represented as a *graph*:
- **Nodes** = atoms (with features like atomic number, electronegativity)
- **Edges** = bonds between atoms within a cutoff radius (with features like interatomic distance)
- **Periodicity** is handled by including edges to periodic image atoms

### The Landscape of Crystal GNNs

| Model | Year | Key Innovation | Paper |
|-------|------|---------------|-------|
| **CGCNN** | 2018 | First crystal graph CNN | Xie & Grossman, PRL 120, 145301 |
| **MEGNet** | 2019 | Global state + universal framework | Chen et al., Chem. Mater. 31, 3564 |
| **ALIGNN** | 2021 | Line graphs for bond angles | Choudhary & DeCost, npj Comp. Mat. 7, 185 |
| **M3GNet** | 2022 | Universal interatomic potential | Chen & Ong, Nature Comp. Sci. 2, 718 |
| **CHGNet** | 2023 | Charge-informed universal potential | Deng et al., Nature Mach. Intell. 5, 1031 |
| **MACE** | 2023 | Higher-order equivariant messages | Batatia et al., NeurIPS 2022 |

We'll use **MEGNet** (via the `matgl` library) as our primary model — it's simple, well-documented, and CPU-friendly.